In [1]:
import random

In [2]:
#设总共词汇量1000
code_token=[]
code_len=150
add_code_len=200-code_len
for i in range(code_len):
    code_token.append(random.randint(0,1000))
for i in range(add_code_len):
    code_token.append(1)

# 模拟code token有150个单词，填充1至定长200

tree_token=[]
tree_len=30
add_tree_len=50-tree_len
for i in range(tree_len):
    tree_token.append(random.randint(0,1000))
for i in range(add_tree_len):
    tree_token.append(1)

# 模拟sst有30个单词，填充1至定长50

qurie_token=[]
qurie_len=8
add_qurie_len=10-qurie_len
for i in range(qurie_len):
    qurie_token.append(random.randint(0,1000))
for i in range(add_qurie_len):
    qurie_token.append(1)
# 模拟qurie token有8个单词，填充1至定长10

In [3]:
code_token

[780,
 96,
 953,
 339,
 903,
 294,
 865,
 251,
 461,
 328,
 628,
 377,
 824,
 111,
 114,
 175,
 581,
 741,
 307,
 307,
 664,
 142,
 682,
 672,
 700,
 758,
 420,
 722,
 168,
 262,
 386,
 455,
 80,
 95,
 945,
 966,
 953,
 833,
 406,
 468,
 749,
 739,
 76,
 382,
 636,
 839,
 902,
 764,
 170,
 110,
 901,
 191,
 563,
 831,
 936,
 160,
 111,
 883,
 38,
 892,
 103,
 121,
 740,
 8,
 651,
 954,
 823,
 568,
 198,
 7,
 344,
 561,
 927,
 0,
 389,
 735,
 536,
 254,
 859,
 550,
 494,
 367,
 986,
 443,
 121,
 73,
 561,
 60,
 907,
 982,
 133,
 508,
 273,
 453,
 614,
 536,
 479,
 604,
 107,
 206,
 921,
 980,
 34,
 944,
 754,
 767,
 336,
 360,
 539,
 353,
 67,
 688,
 712,
 861,
 649,
 963,
 358,
 908,
 891,
 864,
 630,
 529,
 158,
 263,
 277,
 319,
 217,
 664,
 980,
 607,
 106,
 154,
 16,
 905,
 297,
 564,
 940,
 980,
 630,
 398,
 392,
 534,
 635,
 663,
 731,
 653,
 513,
 802,
 160,
 175,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,


In [4]:
qurie_token

[938, 543, 942, 852, 642, 753, 822, 287, 1, 1]

In [5]:
tree_token

[211,
 240,
 132,
 869,
 319,
 820,
 217,
 851,
 955,
 404,
 159,
 390,
 923,
 545,
 475,
 375,
 602,
 272,
 633,
 499,
 873,
 856,
 312,
 744,
 152,
 638,
 38,
 175,
 267,
 470,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1]

In [6]:
import tensorflow as tf
import numpy as np

In [7]:
#这一段是利用的transfrom前的encode部分作为code，tree，与quire的codeer
def scaled_dot_product_attention(q, k, v, mask):
  matmul_qk = tf.matmul(q, k, transpose_b=True)  
  dk = tf.cast(tf.shape(k)[-1], tf.float32)
  scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)
  if mask is not None:
    scaled_attention_logits += (mask * -1e9)
  attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1) 
  output = tf.matmul(attention_weights, v) 
  return output, attention_weights
class MultiHeadAttention(tf.keras.layers.Layer):
  def __init__(self, d_model, num_heads):
    super(MultiHeadAttention, self).__init__()
    self.num_heads = num_heads
    self.d_model = d_model

    assert d_model % self.num_heads == 0

    self.depth = d_model // self.num_heads

    self.wq = tf.keras.layers.Dense(d_model)
    self.wk = tf.keras.layers.Dense(d_model)
    self.wv = tf.keras.layers.Dense(d_model)

    self.dense = tf.keras.layers.Dense(d_model)

  def split_heads(self, x, batch_size):
    x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
    return tf.transpose(x, perm=[0, 2, 1, 3])

  def call(self, v, k, q, mask):
    batch_size = tf.shape(q)[0]

    q = self.wq(q) 
    k = self.wk(k)
    v = self.wv(v) 

    q = self.split_heads(q, batch_size)  
    k = self.split_heads(k, batch_size) 
    v = self.split_heads(v, batch_size)  

    scaled_attention, attention_weights = scaled_dot_product_attention(
        q, k, v, mask)

    scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])  

    concat_attention = tf.reshape(scaled_attention,
                                  (batch_size, -1, self.d_model)) 

    output = self.dense(concat_attention) 

    return output, attention_weights
def point_wise_feed_forward_network(d_model, dff):
  return tf.keras.Sequential([
      tf.keras.layers.Dense(dff, activation='relu'),  
      tf.keras.layers.Dense(d_model) 
  ])
class EncoderLayer(tf.keras.layers.Layer):
  def __init__(self, d_model, num_heads, dff, rate=0.1):
    super(EncoderLayer, self).__init__()

    self.mha = MultiHeadAttention(d_model, num_heads)
    self.ffn = point_wise_feed_forward_network(d_model, dff)

    self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
    self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    self.dropout1 = tf.keras.layers.Dropout(rate)
    self.dropout2 = tf.keras.layers.Dropout(rate)

  def call(self, x, training, mask):

    attn_output, _ = self.mha(x, x, x, mask)  
    attn_output = self.dropout1(attn_output, training=training)
    out1 = self.layernorm1(x + attn_output)  

    ffn_output = self.ffn(out1)  
    ffn_output = self.dropout2(ffn_output, training=training)
    out2 = self.layernorm2(out1 + ffn_output) 

    return out2
def get_angles(pos, i, d_model):
    angle_rates = 1 / np.power(10000, (2 * (i//2)) / np.float32(d_model))
    return pos * angle_rates
def positional_encoding(position, d_model):
    angle_rads = get_angles(np.arange(position)[:, np.newaxis],
                          np.arange(d_model)[np.newaxis, :],
                          d_model)
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
 
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    pos_encoding = angle_rads[np.newaxis, ...]

    return tf.cast(pos_encoding, dtype=tf.float32)
class Encoder(tf.keras.layers.Layer):
  def __init__(self, num_layers, d_model, num_heads, dff, input_vocab_size,
               maximum_position_encoding, rate=0.1):
    super(Encoder, self).__init__()

    self.d_model = d_model
    self.num_layers = num_layers

    self.embedding = tf.keras.layers.Embedding(input_vocab_size, d_model)
    self.pos_encoding = positional_encoding(maximum_position_encoding,
                                            self.d_model)

    self.enc_layers = [EncoderLayer(d_model, num_heads, dff, rate)
                       for _ in range(num_layers)]

    self.dropout = tf.keras.layers.Dropout(rate)

  def call(self, x, training, mask):

    seq_len = tf.shape(x)[1]

    x = self.embedding(x) 
    x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
    x += self.pos_encoding[:, :seq_len, :]

    x = self.dropout(x, training=training)

    for i in range(self.num_layers):
      x = self.enc_layers[i](x, training, mask)
    
    out=[0.0*i for i in range(self.d_model)]
    out=tf.expand_dims(tf.convert_to_tensor(out),0)
    
    for i in tf.unstack(x,axis=1):
        out=out+i
    return out  

In [8]:
code_encoder = Encoder(num_layers=2, d_model=512, num_heads=8,
                         dff=2048, input_vocab_size=1001,
                         maximum_position_encoding=10000)

tree_encoder = Encoder(num_layers=2, d_model=512, num_heads=8,
                         dff=2048, input_vocab_size=1001,
                         maximum_position_encoding=10000)

quire_encoder = Encoder(num_layers=2, d_model=512, num_heads=8,
                         dff=2048, input_vocab_size=1001,
                         maximum_position_encoding=10000)

In [9]:
@tf.function
def tf_cosine_distance(tensor1,tensor2):
    tf.cast(tf.convert_to_tensor(tensor1),tf.bfloat16)
    tf.cast(tf.convert_to_tensor(tensor2),tf.bfloat16)
    tensor1_norm = tf.sqrt(tf.reduce_sum(tf.square(tensor1)))
    tensor2_norm = tf.sqrt(tf.reduce_sum(tf.square(tensor2)))
    tensor1_tensor2 = tf.reduce_sum(tf.multiply(tensor1,tensor2))
    cosin = tensor1_tensor2/(tensor1_norm*tensor2_norm)
    return cosin

In [10]:
from tensorflow.keras import layers,datasets,optimizers,losses

In [11]:
tree_token=tf.expand_dims(tf.convert_to_tensor(tree_token),axis=0)

In [12]:
code_token=tf.expand_dims(tf.convert_to_tensor(code_token),axis=0)

In [13]:
qurie_token=tf.expand_dims(tf.convert_to_tensor(qurie_token),axis=0)

In [14]:
qurie_token

<tf.Tensor: shape=(1, 10), dtype=int32, numpy=array([[938, 543, 942, 852, 642, 753, 822, 287,   1,   1]])>

In [15]:
optimizer=optimizers.Adam(lr=0.0001)
for i in range(5):
    with tf.GradientTape() as tape:
            tree=tree_encoder(tree_token, mask=None)
            code=code_encoder(code_token, mask=None)
            quire=quire_encoder(qurie_token, mask=None)
            out=tree+code
            loss=tf_cosine_distance(out,quire)
    grad=tape.gradient(loss,code_encoder.trainable_variables+tree_encoder.trainable_variables+quire_encoder.trainable_variables)
    optimizer.apply_gradients(zip(grad,code_encoder.trainable_variables+tree_encoder.trainable_variables+quire_encoder.trainable_variables))
    
    print( 'lose', float(loss.numpy()))


D:\Program Files\Anaconda3\lib\site-packages\keras\optimizer_v2\adam.py:105: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(Adam, self).__init__(name, **kwargs)


lose 0.04914356395602226
lose -0.475442498922348
lose -0.7832149267196655
lose -0.9069697260856628
lose -0.9444338083267212
